# Imputation Technique 4: Mean / Median Imputation

**Dataset:** `Loan_Default.csv`

**When to use:** For **numerical** features. Assumes data is **Missing At Random (MCAR)**.

**Key concept:**
- **Mean:** Replace NaN with the arithmetic average. Best for **normally distributed** data.
- **Median:** Replace NaN with the middle value. Best for **skewed** distributions or when **outliers** are present.

---

### Step 1: Setup — Data Loading & Prep

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_prep = df.copy()
for c in Nominal_features:
    df_prep[c + '_freq'] = df_prep[c].map(df_prep.groupby(c).size() / df_prep.shape[0])
    indexer = pd.factorize(df_prep[c], sort=True)[1]
    df_prep[c] = indexer.get_indexer(df_prep[c])
df_prep = df_prep.drop(Nominal_features, axis=1)

high_missing_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'property_value', 'LTV', 'dtir1']
df_drop = df_prep.drop(high_missing_cols, axis=1)

print(f'Starting shape: {df_drop.shape}')

### Step 2: Define Columns to Impute

In [ ]:
Cols_to_be_imputed = [
    'term', 'income', 'age',
    'loan_limit_freq', 'approv_in_adv_freq', 'loan_purpose_freq',
    'Neg_ammortization_freq', 'submission_of_application_freq'
]

### Step 3: Apply Mean Imputation

Using a manual loop with `fillna(col.mean())`.

In [ ]:
df_mean = df_drop.copy()

for c in Cols_to_be_imputed:
    df_mean[c].fillna(df_mean[c].mean(), inplace=True)

df_mean.head()

### Step 4: Apply Median Imputation (Alternative)

Better for skewed data like `income`.

In [ ]:
df_median = df_drop.copy()

for c in Cols_to_be_imputed:
    df_median[c].fillna(df_median[c].median(), inplace=True)

print(f"Mean of income:   {df_drop['income'].mean():.2f}")
print(f"Median of income: {df_drop['income'].median():.2f}")

### Step 5: Verify Results

In [ ]:
missing_after = df_mean.isna().sum()
print('Missing values after Mean Imputation:')
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'None — all filled!')